In [4]:
# --------------------------
# TERM & UNIFICATION SUPPORT
# --------------------------

def is_variable(x):
    return isinstance(x, str) and x[0].islower()

def unify(x, y, theta={}):
    if theta is None:
        return None
    elif x == y:
        return theta
    elif is_variable(x):
        return unify_var(x, y, theta)
    elif is_variable(y):
        return unify_var(y, x, theta)
    elif isinstance(x, tuple) and isinstance(y, tuple):
        if x[0] != y[0] or len(x) != len(y):
            return None
        return unify(x[1:], y[1:], unify(x[0], y[0], theta))
    else:
        return None

def unify_var(var, x, theta):
    if var in theta:
        return unify(theta[var], x, theta)
    elif x in theta:
        return unify(var, theta[x], theta)
    elif occurs_check(var, x, theta):
        return None
    else:
        theta2 = theta.copy()
        theta2[var] = x
        return theta2

def occurs_check(var, x, theta):
    if var == x:
        return True
    if isinstance(x, tuple):
        return any(occurs_check(var, arg, theta) for arg in x[1:])
    if x in theta:
        return occurs_check(var, theta[x], theta)
    return False


# --------------------------
# RESOLUTION SUPPORT
# --------------------------

def apply_substitution(theta, clause):
    new_clause = []
    for pred, args in clause:
        new_args = []
        for a in args:
            while a in theta:
                a = theta[a]
            new_args.append(a)
        new_clause.append((pred, tuple(new_args)))
    return frozenset(new_clause)


def resolve(ci, cj):
    """Return resolvents and show concise info."""
    resolvents = set()

    for lit1 in ci:
        for lit2 in cj:
            if lit1[0] != lit2[0] and lit1[1] == lit2[1]:
                theta = unify(lit1[1], lit2[1], {})
                if theta is not None:
                    new_clause = (ci - {lit1}) | (cj - {lit2})
                    new_clause = apply_substitution(theta, new_clause)

                    print("     Unifier:", theta)
                    print("     Resolvent:", new_clause)

                    resolvents.add(new_clause)

    return resolvents


# --------------------------
# RESOLUTION MAIN FUNCTION
# --------------------------

def resolution(kb, query):
    not_query = frozenset({("¬Likes", ("John", "Peanuts"))})
    clauses = kb + [not_query]

    print("\n===== INITIAL CLAUSES =====")
    for c in clauses:
        print(" ", c)

    new = set()
    step = 1

    while True:
        print(f"\n===== STEP {step} =====")
        step += 1

        pairs = [(clauses[i], clauses[j])
                 for i in range(len(clauses))
                 for j in range(i + 1, len(clauses))]

        for (ci, cj) in pairs:
            print("\n Resolving:")
            print("   ", ci)
            print("   ", cj)

            resolvents = resolve(ci, cj)

            if frozenset() in resolvents:
                print("\n>>> EMPTY CLAUSE FOUND — QUERY PROVED!")
                return True

            new |= resolvents

        if new.issubset(clauses):
            print("\n>>> No new clauses — QUERY CANNOT BE PROVED.")
            return False

        for r in new:
            if r not in clauses:
                print("  New clause added:", r)

        clauses.extend(list(new))


# --------------------------
# KNOWLEDGE BASE
# --------------------------

KB = [
    frozenset({("¬Food", ("x",)), ("Likes", ("John", "x"))}),
    frozenset({("Food", ("Apple",))}),
    frozenset({("Food", ("Vegetable",))}),
    frozenset({("¬Eats", ("x", "y")), ("Killed", ("x",)), ("Food", ("y",))}),
    frozenset({("Eats", ("Anil", "Peanuts"))}),
    frozenset({("Alive", ("Anil",))}),
    frozenset({("¬Eats", ("Anil", "y")), ("Eats", ("Harry", "y"))}),
    frozenset({("¬Alive", ("x",)), ("¬Killed", ("x",))}),
    frozenset({("Killed", ("x",)), ("Alive", ("x",))})
]

# --------------------------
# RUN QUERY
# --------------------------

query = ("Likes", ("John", "Peanuts"))

result = resolution(KB, query)
print("\nFINAL RESULT:", "PROVED" if result else "NOT PROVED")



===== INITIAL CLAUSES =====
  frozenset({('¬Food', ('x',)), ('Likes', ('John', 'x'))})
  frozenset({('Food', ('Apple',))})
  frozenset({('Food', ('Vegetable',))})
  frozenset({('¬Eats', ('x', 'y')), ('Food', ('y',)), ('Killed', ('x',))})
  frozenset({('Eats', ('Anil', 'Peanuts'))})
  frozenset({('Alive', ('Anil',))})
  frozenset({('Eats', ('Harry', 'y')), ('¬Eats', ('Anil', 'y'))})
  frozenset({('¬Alive', ('x',)), ('¬Killed', ('x',))})
  frozenset({('Alive', ('x',)), ('Killed', ('x',))})
  frozenset({('¬Likes', ('John', 'Peanuts'))})

===== STEP 1 =====

 Resolving:
    frozenset({('¬Food', ('x',)), ('Likes', ('John', 'x'))})
    frozenset({('Food', ('Apple',))})

 Resolving:
    frozenset({('¬Food', ('x',)), ('Likes', ('John', 'x'))})
    frozenset({('Food', ('Vegetable',))})

 Resolving:
    frozenset({('¬Food', ('x',)), ('Likes', ('John', 'x'))})
    frozenset({('¬Eats', ('x', 'y')), ('Food', ('y',)), ('Killed', ('x',))})
     Unifier: {}
     Resolvent: frozenset({('¬Eats', ('x', 